In [0]:
dbutils.widgets.removeAll()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
dbutils.widgets.text("container", "raw")
dbutils.widgets.text("catalogo", "catalog_clinica")
dbutils.widgets.text("esquema", "bronze")
dbutils.widgets.text("storageName", "adlsproyecto2404")

In [0]:
container = dbutils.widgets.get("container")
catalogo = dbutils.widgets.get("catalogo")
esquema = dbutils.widgets.get("esquema")
storageName = dbutils.widgets.get("storageName")

ruta = f"abfss://{container}@{storageName}.dfs.core.windows.net/pagos.csv"

In [0]:
pagos_schema = StructType(fields=[
    StructField("id_pago", IntegerType(), True),
    StructField("id_atencion", IntegerType(), True),
    StructField("id_cita", IntegerType(), True),
    StructField("monto_total", IntegerType(), True),
    StructField("descuento", IntegerType(), True),
    StructField("monto_pagado", IntegerType(), True),
    StructField("tipo_pago", StringType(), True),
    StructField("estado_pago", StringType(), True)
])

In [0]:
df_pagos_read = spark.read.option('header', True)\
                          .schema(pagos_schema)\
                          .csv(ruta)

In [0]:
df_pagos_ingestion = df_pagos_read.withColumn("INGESTION_DATE", current_timestamp())

In [0]:
df_pagos_final = df_pagos_ingestion.select(
    "id_pago",
    "id_atencion",
    "id_cita",
    "monto_total",
    "descuento",
    "monto_pagado",
    "tipo_pago",
    "estado_pago",
    "INGESTION_DATE"
)

In [0]:
df_pagos_final.write.mode("overwrite").insertInto(f"{catalogo}.{esquema}.pagos")